In [ ]:
!pip install kagglehub

In [ ]:
!pip install -q -U kaggle


In [ ]:
# STEP 1: Extract arXiv dataset from Kaggle and filter Computer Science papers
# related to AI-focused areas (ML, NLP, Computer Vision, etc.) from 2021–2026
# Create a balanced dataset (3000 papers, equally distributed per year)
# for trend analysis over time

import os
os.environ["KAGGLE_API_TOKEN"] = "KAGGle API Token"

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d Cornell-University/arxiv

In [ ]:
!ls

In [ ]:
!unzip arxiv.zip -d arxiv_data

In [ ]:
import json
import pandas as pd
from datetime import datetime

FILE_PATH = "arxiv_data/arxiv-metadata-oai-snapshot.json"

TARGET_TOTAL = 3000
START_YEAR = 2021
END_YEAR = 2026

TARGET_CATEGORIES = [
    "cs.AI", #Artificial Intelligence
    "cs.CL", # Computation and Language
    "cs.LG", #Machine Learning
    "cs.CV", #Computer Vision and Pattern Recognition
    "cs.NE", #Neural and Evolutionary Computing
    "cs.IR", #Information Retrieval
]

print(f"Extracting papers from {START_YEAR} to {END_YEAR}...")

data = []

with open(FILE_PATH, "r") as f:
    for i, line in enumerate(f):
        try:
            paper = json.loads(line)

            categories = paper.get("categories", "")
            category_list = categories.split()

            if not any(cat in category_list for cat in TARGET_CATEGORIES):
                continue

            title = paper.get("title", "").strip()
            abstract = paper.get("abstract", "").strip()
            authors = paper.get("authors", "").strip()

            if not title or not abstract:
                continue

            versions = paper.get("versions", [])
            if not versions:
                continue

            created_str = versions[0].get("created", "").strip()
            if not created_str:
                continue

            try:
                created_date = datetime.strptime(created_str, "%a, %d %b %Y %H:%M:%S %Z")
            except:
                continue

            year = created_date.year
            month = created_date.month
            year_month = f"{year}-{month:02d}"

            if year < START_YEAR or year > END_YEAR:
                continue

            data.append({
                "id": paper.get("id", ""),
                "title": title,
                "abstract": abstract,
                "categories": categories,
                "authors": authors,
                "year": year,
                "month": month,
                "year_month": year_month
            })

            if (i + 1) % 50000 == 0:
                print(f"Scanned {i+1:,} lines | Matched so far: {len(data):,}")

        except:
            continue

print("\nTotal matched papers:", len(data))

df = pd.DataFrame(data)

print("\nPapers per year:")
print(df["year"].value_counts().sort_index())

In [ ]:
TARGET_TOTAL = 3000

samples_per_year = TARGET_TOTAL // 6   # 2021, 2022, 2023, 2024, 2025, 2026

df_balanced = (
    df.groupby("year", group_keys=False)
      .apply(lambda x: x.sample(n=samples_per_year, random_state=42))
      .reset_index(drop=True)
)

df_balanced = df_balanced.sort_values(["year", "month"]).reset_index(drop=True)

print("Final dataset size:", len(df_balanced))
print("\nPapers per year after sampling:")
print(df_balanced["year"].value_counts().sort_index())

In [ ]:
df_balanced["text"] = df_balanced["title"] + " " + df_balanced["abstract"]
df_balanced.head()

In [ ]:
df_balanced.to_csv("arxiv_project_dataset_3000.csv", index=False)
print("Saved successfully!")
from google.colab import files
files.download("arxiv_project_dataset_3000.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Load CSV
import pandas as pd
df_balanced = pd.read_csv("/content/drive/Shareddrives/CMPE 255/arxiv_project_dataset_3000.csv")
print(df_balanced.shape)
df_balanced.head()
df = df_balanced.copy()

## Data Warehouse Design

This project uses a flat-file data warehouse approach stored in Google Drive.

### Schema Design (Star Schema)

**Fact Table: papers_fact**
- paper_id (PK)
- year
- month
- topic_id (FK)
- abstract_length

**Dimension Tables:**
- dim_category: category_id, category_name, category_code
- dim_time: year, month, quarter, year_month
- dim_topic: topic_id, topic_name, top_keywords

### ETL Process
1. Extract — Downloaded raw JSON from Kaggle (arXiv dataset)
2. Transform — Filtered by category and year, balanced sampling
3. Load — Saved as CSV to Google Shared Drive

In [ ]:
# DATA WAREHOUSE - Schema Visualization

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

# --- Fact Table (center) ---
fact = mpatches.FancyBboxPatch((3.5, 2.5), 3, 3.5,
    boxstyle="round,pad=0.15",
    linewidth=3, edgecolor='#00d4ff', facecolor='#16213e')
ax.add_patch(fact)
ax.text(5, 5.65, 'FACT TABLE', ha='center', fontsize=12,
        fontweight='bold', color='#00d4ff')
ax.text(5, 5.2, 'papers_fact', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(5, 4.85, '─────────────', ha='center', fontsize=8, color='#00d4ff')
ax.text(5, 4.45, '• paper_id (PK)', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(5, 4.05, '• year', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(5, 3.65, '• month', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(5, 3.25, '• topic_id (FK)', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(5, 2.85, '• abstract_length', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')

# --- Dimension Table 1 (left) ---
dim1 = mpatches.FancyBboxPatch((0.2, 3.8), 2.8, 2.8,
    boxstyle="round,pad=0.15",
    linewidth=3, edgecolor='#00ff88', facecolor='#16213e')
ax.add_patch(dim1)
ax.text(1.6, 6.25, 'dim_category', ha='center', fontsize=11,
        fontweight='bold', color='#00ff88')
ax.text(1.6, 5.85, '─────────────', ha='center', fontsize=8, color='#00ff88')
ax.text(1.6, 5.45, '• category_id (PK)', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(1.6, 5.05, '• category_name', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(1.6, 4.65, '• category_code', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')

# --- Dimension Table 2 (top) ---
dim2 = mpatches.FancyBboxPatch((3.5, 6.2), 3, 1.6,
    boxstyle="round,pad=0.15",
    linewidth=3, edgecolor='#ff00ff', facecolor='#16213e')
ax.add_patch(dim2)
ax.text(5, 7.45, 'dim_time', ha='center', fontsize=11,
        fontweight='bold', color='#ff00ff')
ax.text(5, 7.0, '─────────────', ha='center', fontsize=8, color='#ff00ff')
ax.text(5, 6.55, '• year   • month   • quarter', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')

# --- Dimension Table 3 (right) ---
dim3 = mpatches.FancyBboxPatch((7, 3.8), 2.8, 2.8,
    boxstyle="round,pad=0.15",
    linewidth=3, edgecolor='#ff6b00', facecolor='#16213e')
ax.add_patch(dim3)
ax.text(8.4, 6.25, 'dim_topic', ha='center', fontsize=11,
        fontweight='bold', color='#ff6b00')
ax.text(8.4, 5.85, '─────────────', ha='center', fontsize=8, color='#ff6b00')
ax.text(8.4, 5.45, '• topic_id (PK)', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(8.4, 5.05, '• topic_name', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')
ax.text(8.4, 4.65, '• top_keywords', ha='center', fontsize=10,
        color='#ffffff', fontweight='bold')

# --- ETL Box (bottom) ---
etl = mpatches.FancyBboxPatch((1.5, 0.3), 7, 1.6,
    boxstyle="round,pad=0.15",
    linewidth=3, edgecolor='#ffd700', facecolor='#16213e')
ax.add_patch(etl)
ax.text(5, 1.6, 'ETL PIPELINE', ha='center', fontsize=12,
        fontweight='bold', color='#ffd700')
ax.text(5, 1.1,
        'Extract (Kaggle JSON)  →  Transform (Filter + Clean)  →  Load (Google Drive CSV)',
        ha='center', fontsize=10, color='#ffffff', fontweight='bold')

# --- Arrows ---
ax.annotate('', xy=(3.5, 4.8), xytext=(3.0, 5.2),
            arrowprops=dict(arrowstyle='->', color='#00ff88', lw=2.5))
ax.annotate('', xy=(5, 6.2), xytext=(5, 5.9),
            arrowprops=dict(arrowstyle='->', color='#ff00ff', lw=2.5))
ax.annotate('', xy=(7, 4.8), xytext=(6.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='#ff6b00', lw=2.5))
ax.annotate('', xy=(5, 2.5), xytext=(5, 1.9),
            arrowprops=dict(arrowstyle='->', color='#ffd700', lw=2.5))

plt.title("Data Warehouse Star Schema", fontsize=18,
          fontweight='bold', color='#ffffff', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Chart 1: Papers per Year ---
df_balanced['year'].value_counts().sort_index().plot(kind='bar', color='steelblue', figsize=(8, 4))
plt.title("Papers per Year (Raw)")
plt.xlabel("Year")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# --- Chart 2: Papers per Target Category (clean version) ---
TARGET_CATEGORIES = ["cs.AI", "cs.CL", "cs.LG", "cs.CV", "cs.NE", "cs.IR"]

cat_counts = {}
for cat in TARGET_CATEGORIES:
    cat_counts[cat] = df_balanced['categories'].str.contains(cat).sum()

pd.Series(cat_counts).plot(kind='bar', color='coral', figsize=(8, 4))
plt.title("Papers per Target Category (Raw)")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Chart 3: Abstract Length Distribution ---
df_balanced['abstract_len'] = df_balanced['abstract'].apply(len)
df_balanced['abstract_len'].hist(bins=40, color='purple', figsize=(8, 4))
plt.title("Abstract Length Distribution (Raw)")
plt.xlabel("Character Count")
plt.ylabel("Number of Papers")
plt.tight_layout()
plt.show()

# --- Chart 4: Papers per Year AND Category (heatmap) ---
cat_year = pd.DataFrame({
    cat: df_balanced[df_balanced['categories'].str.contains(cat)]['year'].value_counts().sort_index()
    for cat in TARGET_CATEGORIES
})
sns.heatmap(cat_year, annot=True, fmt='d', cmap='Blues')
plt.title("Papers per Year per Category (Raw)")
plt.tight_layout()
plt.show()

In [ ]:
# BEFORE vs AFTER TEXT CLEANING VISUALIZATION

import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ---- BEFORE CLEANING ----
raw_text = ' '.join(df_balanced['text'])
df_balanced['raw_len'] = df_balanced['text'].apply(lambda x: len(x.split()))

# ---- DEFINE CLEANING FUNCTION ----
CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS) | {
    "paper", "propose", "proposed", "method", "model", "approach",
    "show", "result", "results", "based", "using", "used", "use",
    "task", "dataset", "training", "trained", "train", "learn",
    "learning", "performance", "achieve", "achieved", "state", "art",
    "existing", "work", "works", "problem", "demonstrate", "demonstrates",
    "also", "study", "studies", "new", "novel", "improve", "improved",
    "system", "different", "large", "experiments", "experimental",
    "evaluation", "evaluate", "deep", "network", "networks"
}

KEEP_TERMS = {
    "transformer", "llm", "gpt", "bert", "attention", "diffusion",
    "generation", "language", "retrieval", "reasoning", "fine", "tuning",
    "prompt", "alignment", "multimodal", "vision", "embedding"
}
CUSTOM_STOPWORDS = CUSTOM_STOPWORDS - KEEP_TERMS

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\$.*?\$', ' ', text)
    text = re.sub(r'\\[a-z]+\{.*?\}', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b\w{1,2}\b', ' ', text)
    words = text.split()
    words = [w for w in words if w not in CUSTOM_STOPWORDS]
    return ' '.join(words)

# ---- AFTER CLEANING ----
df_balanced['clean_text_preview'] = df_balanced['text'].apply(clean_text)
clean_text_all = ' '.join(df_balanced['clean_text_preview'])
df_balanced['clean_len'] = df_balanced['clean_text_preview'].apply(lambda x: len(x.split()))

# ---- PLOT 1: Word Clouds side by side ----
wc_before = WordCloud(width=800, height=400,
                      background_color='white',
                      colormap='Reds').generate(raw_text)

wc_after = WordCloud(width=800, height=400,
                     background_color='white',
                     colormap='Blues').generate(clean_text_all)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(wc_before, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title("Word Cloud BEFORE Cleaning", fontsize=14, fontweight='bold')

axes[1].imshow(wc_after, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title("Word Cloud AFTER Cleaning", fontsize=14, fontweight='bold')

plt.suptitle("Text Before vs After Preprocessing", fontsize=16)
plt.tight_layout()
plt.show()

# ---- PLOT 2: Word Count Distribution side by side ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_balanced['raw_len'], bins=40, color='tomato', edgecolor='black')
axes[0].set_title("Word Count BEFORE Cleaning", fontweight='bold')
axes[0].set_xlabel("Number of Words")
axes[0].set_ylabel("Number of Papers")

axes[1].hist(df_balanced['clean_len'], bins=40, color='steelblue', edgecolor='black')
axes[1].set_title("Word Count AFTER Cleaning", fontweight='bold')
axes[1].set_xlabel("Number of Words")
axes[1].set_ylabel("Number of Papers")

plt.suptitle("Word Count Distribution Before vs After Cleaning", fontsize=14)
plt.tight_layout()
plt.show()

# ---- PLOT 3: Average word count comparison ----
avg_before = df_balanced['raw_len'].mean()
avg_after = df_balanced['clean_len'].mean()

plt.figure(figsize=(6, 4))
plt.bar(['Before Cleaning', 'After Cleaning'],
        [avg_before, avg_after],
        color=['tomato', 'steelblue'],
        edgecolor='black')
plt.title("Average Words per Paper: Before vs After", fontweight='bold')
plt.ylabel("Average Word Count")
for i, v in enumerate([avg_before, avg_after]):
    plt.text(i, v + 1, f'{v:.0f} words', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Average words BEFORE cleaning: {avg_before:.0f}")
print(f"Average words AFTER cleaning:  {avg_after:.0f}")
print(f"Words removed on average: {avg_before - avg_after:.0f} ({((avg_before-avg_after)/avg_before)*100:.1f}%)")

In [ ]:
# STEP 2: Preprocess text
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Expand stopwords with domain-specific noise words
CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS) | {
    "paper", "propose", "proposed", "method", "model", "approach",
    "show", "result", "results", "based", "using", "used", "use",
    "task", "dataset", "training", "trained", "train", "learn",
    "learning", "performance", "achieve", "achieved", "state", "art",
    "existing", "work", "works", "problem", "demonstrate", "demonstrates",
    "also", "study", "studies", "new", "novel", "improve", "improved",
    "system", "different", "large", "experiments", "experimental",
    "evaluation", "evaluate", "deep", "network", "networks"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\$.*?\$', ' ', text)           # remove LaTeX math
    text = re.sub(r'\\[a-z]+\{.*?\}', ' ', text)   # remove LaTeX commands
    text = re.sub(r'[^a-z\s]', ' ', text)           # keep only letters
    text = re.sub(r'\b\w{1,2}\b', ' ', text)        # remove 1-2 char words
    words = text.split()
    words = [w for w in words if w not in CUSTOM_STOPWORDS]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)
print(df['clean_text'].iloc[0][:300])

In [ ]:
!pip install bertopic sentence-transformers

In [ ]:
# STEP 3: Train BERTopic with better topic separation

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS) | {
    "paper", "propose", "proposed", "method", "model", "approach",
    "show", "result", "results", "based", "using", "used", "use",
    "task", "dataset", "training", "trained", "train", "learn",
    "learning", "performance", "achieve", "achieved", "state", "art",
    "existing", "work", "works", "problem", "demonstrate", "demonstrates",
    "also", "study", "studies", "new", "novel", "improve", "improved",
    "system", "different", "large", "experiments", "experimental",
    "evaluation", "evaluate", "deep", "network", "networks"
}

# Make sure key AI terms are NOT removed
KEEP_TERMS = {
    "transformer", "llm", "gpt", "bert", "attention", "diffusion",
    "generation", "language", "retrieval", "reasoning", "fine", "tuning",
    "prompt", "alignment", "multimodal", "vision", "embedding"
}
CUSTOM_STOPWORDS = CUSTOM_STOPWORDS - KEEP_TERMS

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\$.*?\$', ' ', text)
    text = re.sub(r'\\[a-z]+\{.*?\}', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b\w{1,2}\b', ' ', text)
    words = text.split()
    words = [w for w in words if w not in CUSTOM_STOPWORDS]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)
docs = df['clean_text'].tolist()

vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.85,
    ngram_range=(1, 2)
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fix randomness with random_state=42
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    random_state=42  # ← fixes topic order every run
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,       # ← added
    hdbscan_model=hdbscan_model, # ← added
    language="english",
    calculate_probabilities=True,
    verbose=True,
    nr_topics=20,
    min_topic_size=10
)

topics, probs = topic_model.fit_transform(docs)
df['topic'] = topics

print(f"Number of topics: {len(set(topics)) - 1}")
print(topic_model.get_topic_info().head(20))

In [ ]:
# BERTOPIC RESULTS VISUALIZATION

import matplotlib.pyplot as plt
import pandas as pd

# --- Chart 1: Number of papers per topic ---
topic_info = topic_model.get_topic_info()
topic_info_clean = topic_info[topic_info['Topic'] != -1]

plt.figure(figsize=(12, 5))
plt.bar(topic_info_clean['Topic'].astype(str),
        topic_info_clean['Count'],
        color='steelblue', edgecolor='black')
plt.title("Number of Papers per Topic (After BERTopic)", fontweight='bold', fontsize=14)
plt.xlabel("Topic ID")
plt.ylabel("Number of Papers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Chart 2: Outlier pie chart ---
total = len(df)
outliers = (df['topic'] == -1).sum()
non_outliers = total - outliers

plt.figure(figsize=(6, 6))
plt.pie([non_outliers, outliers],
        labels=['Assigned to Topic', 'Outliers (topic -1)'],
        colors=['steelblue', 'tomato'],
        autopct='%1.1f%%',
        startangle=90)
plt.title("Papers Assigned to Topics vs Outliers", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Total papers: {total}")
print(f"Assigned to topics: {non_outliers} ({non_outliers/total*100:.1f}%)")
print(f"Outliers: {outliers} ({outliers/total*100:.1f}%)")
print(f"Total topics found: {len(topic_info_clean)}")
print("\nTop 5 topics:")
for _, row in topic_info_clean.head(5).iterrows():
    words = topic_model.get_topic(row['Topic'])
    top_words = ", ".join([w for w, _ in words[:5]])
    print(f"  Topic {row['Topic']} ({row['Count']} papers): {top_words}")

In [ ]:
# STEP 4: Visualize topics

# 4a. Bar chart of top words per topic
topic_model.visualize_barchart(top_n_topics=12, n_words=8)

In [ ]:
# 4b. Heatmap of topic similarity
topic_model.visualize_heatmap()

In [ ]:
# MODEL EVALUATION

import matplotlib.pyplot as plt
import numpy as np

# --- Metric 1: Topic Diversity ---
all_words = []
unique_words = set()
for topic_id in topic_info_clean['Topic']:
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    all_words.extend(words)
    unique_words.update(words)

topic_diversity = len(unique_words) / len(all_words)
print(f"Topic Diversity Score: {topic_diversity:.3f}")
print("(1.0 = perfectly unique words per topic, 0 = all topics share same words)")

# --- Metric 2: Topic Size Distribution ---
sizes = topic_info_clean['Count'].tolist()
print(f"\nTopic Size Stats:")
print(f"  Largest topic:  {max(sizes)} papers")
print(f"  Smallest topic: {min(sizes)} papers")
print(f"  Average size:   {np.mean(sizes):.1f} papers")
print(f"  Std deviation:  {np.std(sizes):.1f}")

# --- Metric 3: Outlier Analysis ---
total = len(df)
outliers = (df['topic'] == -1).sum()
print(f"\nOutlier Analysis:")
print(f"  Total papers: {total}")
print(f"  Outliers: {outliers} ({outliers/total*100:.1f}%)")
print(f"  Assigned: {total-outliers} ({(total-outliers)/total*100:.1f}%)")

# --- Chart 1: Keyword Uniqueness per Topic ---
topics_sample = topic_info_clean['Topic'].tolist()[:10]
diversity_per_topic = []
for t in topics_sample:
    words = [w for w, _ in topic_model.get_topic(t)[:10]]
    diversity_per_topic.append(len(set(words)) / len(words))

colors_gradient = plt.cm.viridis(np.linspace(0.3, 0.9, len(topics_sample)))

plt.figure(figsize=(10, 4))
plt.bar([f"Topic {t}" for t in topics_sample],
        diversity_per_topic,
        color=colors_gradient, edgecolor='white', linewidth=0.8)
plt.title("Keyword Uniqueness per Topic", fontweight='bold', fontsize=13)
plt.ylabel("Uniqueness Score")
plt.xlabel("Topic")
plt.xticks(rotation=45)
plt.axhline(y=topic_diversity, color='red', linestyle='--',
            linewidth=1.5, label=f'Average: {topic_diversity:.2f}')
plt.legend()
plt.tight_layout()
plt.show()

# --- Chart 2: Topic Size Distribution ---
plt.figure(figsize=(8, 4))
n, bins, patches = plt.hist(sizes, bins=10, edgecolor='white', linewidth=0.8)
for patch, color in zip(patches, plt.cm.plasma(np.linspace(0.2, 0.85, len(patches)))):
    patch.set_facecolor(color)
plt.title("Topic Size Distribution", fontweight='bold', fontsize=13)
plt.xlabel("Number of Papers in Topic")
plt.ylabel("Number of Topics")
plt.tight_layout()
plt.show()

# --- Chart 3: Evaluation Summary ---
metrics = ['Topic Diversity', 'Assignment Rate', 'Outlier Rate']
values = [topic_diversity * 100, (total-outliers)/total*100, outliers/total*100]
colors_summary = ['#2ecc71', '#3498db', '#e74c3c']

plt.figure(figsize=(8, 4))
bars = plt.bar(metrics, values, color=colors_summary,
               edgecolor='white', linewidth=0.8, width=0.5)
plt.title("Model Evaluation Summary", fontweight='bold', fontsize=13)
plt.ylabel("Percentage / Score (%)")
plt.ylim(0, max(values) + 15)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1.5,
             f'{val:.1f}%',
             ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 5: Quarterly bins for smoother trends
df['quarter'] = df['year'].astype(str) + '-Q' + ((df['month'] - 1) // 3 + 1).astype(str)

timestamps = df['quarter'].tolist()

topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=24
)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=12,
    title="Research Topic Trends (2021–2026, Quarterly)"
)

In [ ]:
df_trend = df[df['year'] < 2026].copy()
docs_trend = df_trend['clean_text'].tolist()
topics_trend = df_trend['topic'].tolist()
df_trend['quarter'] = df_trend['year'].astype(str) + '-Q' + ((df_trend['month'] - 1) // 3 + 1).astype(str)
timestamps_trend = df_trend['quarter'].tolist()

topics_over_time = topic_model.topics_over_time(
    docs_trend,
    timestamps_trend,
    topics=topics_trend,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=20
)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=12,
    title="Research Topic Trends (2021–2025, Quarterly)"
)

In [ ]:
import matplotlib.pyplot as plt

top_topics = topic_model.get_topic_info()
top_topics = top_topics[top_topics['Topic'] != -1].head(8)['Topic'].tolist()

trend_df = df[df['topic'].isin(top_topics)].groupby(['year', 'topic']).size().reset_index(name='count')

# Get topic labels
topic_labels = {
    row['Topic']: row['Name']
    for _, row in topic_model.get_topic_info().iterrows()
}
trend_df['topic_name'] = trend_df['topic'].map(topic_labels)

plt.figure(figsize=(12, 6))
for name, group in trend_df.groupby('topic_name'):
    plt.plot(group['year'], group['count'], marker='o', label=name)

plt.title("Emerging Research Topic Trends (2021–2026)")
plt.xlabel("Year")
plt.ylabel("Number of Papers")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig("topic_trends.png", dpi=150)
plt.show()

In [ ]:
# STEP 6: Summary of findings
info = topic_model.get_topic_info()
info = info[info['Topic'] != -1].head(10)

print("\n═══════════════════════════════════════════")
print("       Top 10 Discovered Topics")
print("═══════════════════════════════════════════\n")
for _, row in info.iterrows():
    words = topic_model.get_topic(row['Topic'])
    top_words = ", ".join([w for w, _ in words[:6]])
    print(f"Topic {row['Topic']} ({row['Count']} papers): {top_words}")

print("\n═══════════════════════════════════════════")
print("           Trend Highlights")
print("═══════════════════════════════════════════\n")
for year in range(2021, 2027):
    year_df = df[df['year'] == year]
    # Exclude outliers topic -1
    year_df_clean = year_df[year_df['topic'] != -1]
    top_topic = year_df_clean['topic'].value_counts().idxmax()
    label = topic_model.get_topic_info(top_topic)['Name'].values[0]
    print(f"{year}: Most dominant topic: {label}")

In [ ]:
# STEP 6 VISUALIZATION

import matplotlib.pyplot as plt
import pandas as pd

# --- Chart 1: Top 10 Topics Bar Chart ---
info = topic_model.get_topic_info()
info = info[info['Topic'] != -1].head(10)

topic_names = [f"Topic {row['Topic']}\n{', '.join([w for w, _ in topic_model.get_topic(row['Topic'])[:3]])}"
               for _, row in info.iterrows()]
topic_counts = info['Count'].tolist()

colors = plt.cm.tab10(range(10))

plt.figure(figsize=(12, 5))
bars = plt.bar(range(len(topic_names)), topic_counts, color=colors, edgecolor='white')
plt.xticks(range(len(topic_names)), topic_names, rotation=45, ha='right', fontsize=8)
plt.title("Top 10 Discovered Topics (Overall)", fontweight='bold', fontsize=14)
plt.ylabel("Number of Papers")
plt.grid(axis='y', alpha=0.3, linestyle='--')
for bar, val in zip(bars, topic_counts):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             str(val), ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Chart 2: Most Dominant Topic Per Year (excluding outliers) ---
dominant_labels = []
dominant_counts = []
dominant_topics_list = []

for year in range(2021, 2027):
    year_df = df[df['year'] == year]
    year_df_clean = year_df[year_df['topic'] != -1]
    top_topic = year_df_clean['topic'].value_counts().idxmax()
    dominant_topics_list.append(top_topic)
    label = topic_model.get_topic_info(top_topic)['Name'].values[0]
    short_label = label.split('_')[1] + '/' + label.split('_')[2]
    dominant_labels.append(short_label)
    dominant_counts.append(year_df_clean['topic'].value_counts().max())

years = list(range(2021, 2027))
year_colors = ['#F8A4A4', '#F4A460', '#F7D060', '#82C882', '#64B5F6', '#B39DDB']

plt.figure(figsize=(10, 6))
bars = plt.bar(years, dominant_counts,
               color=year_colors, edgecolor='white',
               linewidth=1.2, width=0.6)

for bar, label in zip(bars, dominant_labels):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             label, ha='center', fontsize=9,
             fontweight='bold', rotation=0)

plt.title("Most Dominant Topic Per Year (2021–2026)",
          fontweight='bold', fontsize=15, pad=30)
plt.xlabel("Year")
plt.ylabel("Number of Papers in Dominant Topic")
plt.xticks(years)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.ylim(0, max(dominant_counts) + 20)
plt.tight_layout()
plt.show()

# --- Chart 3: LLM vs Computer Vision vs Other Topics Over Years ---
llm_counts = []
cv_counts = []
other_counts = []

# Find LLM topic and CV topic numbers
llm_topic = None
cv_topic = None
for _, row in topic_model.get_topic_info().iterrows():
    if row['Topic'] == -1:
        continue
    name = row['Name'].lower()
    if 'language' in name or 'llm' in name or 'reasoning' in name:
        if llm_topic is None:
            llm_topic = row['Topic']
    if 'image' in name or 'video' in name or 'vision' in name:
        if cv_topic is None:
            cv_topic = row['Topic']

for year in range(2021, 2027):
    year_df = df[(df['year'] == year) & (df['topic'] != -1)]
    llm_counts.append((year_df['topic'] == llm_topic).sum())
    cv_counts.append((year_df['topic'] == cv_topic).sum())
    other_counts.append(len(year_df) -
                       (year_df['topic'] == llm_topic).sum() -
                       (year_df['topic'] == cv_topic).sum())

x = range(len(years))
width = 0.25

llm_color = '#5C6BC0'      # muted indigo blue
cv_color = '#EF7C5A'       # soft terracotta
other_color = '#4CAF82'    # muted sage green


fig, ax = plt.subplots(figsize=(13, 6))

fig.patch.set_facecolor('#FFFFFF')
ax.set_facecolor('#FFFFFF')

bars1 = ax.bar([i - width for i in x], llm_counts, width,
               label='LLM / Language', color=llm_color,
               edgecolor='white', linewidth=1.2)
bars2 = ax.bar(x, cv_counts, width,
               label='Computer Vision', color=cv_color,
               edgecolor='white', linewidth=1.2)
bars3 = ax.bar([i + width for i in x], other_counts, width,
               label='Other Topics', color=other_color,
               edgecolor='white', linewidth=1.2)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            str(int(bar.get_height())),
            ha='center', fontsize=9,
            fontweight='bold', color=llm_color)

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            str(int(bar.get_height())),
            ha='center', fontsize=9,
            fontweight='bold', color=cv_color)

for bar in bars3:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            str(int(bar.get_height())),
            ha='center', fontsize=9,
            fontweight='bold', color=other_color)

ax.set_title("LLM vs Computer Vision vs Other Topics (2021–2026)",
             fontweight='bold', fontsize=14, pad=15)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Number of Papers", fontsize=11)
ax.set_xticks(list(x))
ax.set_xticklabels(years, fontsize=10)
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
!pip install gensim

In [ ]:
# =========================
# BERTopic Evaluation Metrics
# =========================

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# Topic info
topic_info = topic_model.get_topic_info()
topic_info_clean = topic_info[topic_info["Topic"] != -1]

# 1. Outlier Rate and Assignment Rate
total_docs = len(df)
outlier_count = (df["topic"] == -1).sum()
assigned_count = total_docs - outlier_count

outlier_rate = outlier_count / total_docs
assignment_rate = assigned_count / total_docs

print("===== Topic Assignment Metrics =====")
print(f"Total papers: {total_docs}")
print(f"Assigned to topics: {assigned_count} ({assignment_rate:.2%})")
print(f"Outliers: {outlier_count} ({outlier_rate:.2%})")

# 2. Topic Size Statistics
topic_sizes = topic_info_clean["Count"]

print("\n===== Topic Size Metrics =====")
print(f"Number of topics: {len(topic_info_clean)}")
print(f"Largest topic size: {topic_sizes.max()}")
print(f"Smallest topic size: {topic_sizes.min()}")
print(f"Average topic size: {topic_sizes.mean():.2f}")
print(f"Topic size standard deviation: {topic_sizes.std():.2f}")

# 3. Topic Diversity
top_n_words = 10
topic_words = []

for topic_id in topic_info_clean["Topic"]:
    words = topic_model.get_topic(topic_id)
    if words:
        topic_words.extend([word for word, score in words[:top_n_words]])

unique_words = set(topic_words)
topic_diversity = len(unique_words) / len(topic_words)

print("\n===== Topic Diversity =====")
print(f"Topic Diversity Score: {topic_diversity:.3f}")

# 4. Topic Coherence using Gensim
texts = [doc.split() for doc in df["clean_text"]]

dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

topics_words_list = []

for topic_id in topic_info_clean["Topic"]:
    words = topic_model.get_topic(topic_id)
    if words:
        topics_words_list.append([word for word, score in words[:top_n_words]])

coherence_model = CoherenceModel(
    topics=topics_words_list,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()

print("\n===== Topic Coherence =====")
print(f"Coherence Score (c_v): {coherence_score:.3f}")

# 5. Silhouette Score
# Use embeddings only for documents that are assigned to real topics
valid_indices = df["topic"] != -1
valid_topics = df.loc[valid_indices, "topic"]

if valid_topics.nunique() > 1:
    embeddings = embedding_model.encode(df["clean_text"].tolist(), show_progress_bar=True)
    valid_embeddings = embeddings[valid_indices]

    silhouette = silhouette_score(valid_embeddings, valid_topics)
    print("\n===== Cluster Separation =====")
    print(f"Silhouette Score: {silhouette:.3f}")
else:
    print("\nSilhouette Score cannot be calculated because only one topic was found.")

In [ ]:
# =========================
# Evaluation Metrics Visualizations
# =========================

import matplotlib.pyplot as plt
import pandas as pd

# 1. Assignment vs Outlier Pie Chart
plt.figure(figsize=(6, 6))
plt.pie(
    [assigned_count, outlier_count],
    labels=["Assigned to Topics", "Outliers"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Topic Assignment vs Outliers")
plt.tight_layout()
plt.show()


# 2. Topic Size Distribution Bar Chart
plt.figure(figsize=(12, 5))
plt.bar(
    topic_info_clean["Topic"].astype(str),
    topic_info_clean["Count"]
)
plt.title("Number of Papers per Topic")
plt.xlabel("Topic ID")
plt.ylabel("Number of Papers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 3. Evaluation Metrics Summary Bar Chart
metrics_df = pd.DataFrame({
    "Metric": ["Assignment Rate", "Outlier Rate", "Topic Diversity", "Coherence Score"],
    "Score": [assignment_rate, outlier_rate, topic_diversity, coherence_score]
})

plt.figure(figsize=(8, 5))
plt.bar(metrics_df["Metric"], metrics_df["Score"])
plt.title("BERTopic Evaluation Metrics Summary")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# 4. Topic Size Statistics Visualization
topic_stats = pd.DataFrame({
    "Statistic": ["Largest Topic", "Smallest Topic", "Average Topic Size"],
    "Value": [
        topic_sizes.max(),
        topic_sizes.min(),
        topic_sizes.mean()
    ]
})

plt.figure(figsize=(7, 5))
plt.bar(topic_stats["Statistic"], topic_stats["Value"])
plt.title("Topic Size Statistics")
plt.ylabel("Number of Papers")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Silhouette Score Visualization
if valid_topics.nunique() > 1:
    plt.figure(figsize=(5, 4))
    plt.bar(["Silhouette Score"], [silhouette])
    plt.ylim(-1, 1)
    plt.title("Cluster Separation Evaluation")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()